In [24]:
import torch

In [25]:
!wget "https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt"

--2026-09-06 16:10:22--  https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt
Resolving github.com (github.com)... 20.207.73.82
Connecting to github.com (github.com)|20.207.73.82|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘input.txt.5’

input.txt.5             [ <=>                ] 225.13K  --.-KB/s    in 0.1s    

2026-09-06 16:10:22 (1.89 MB/s) - ‘input.txt.5’ saved [230529]



In [26]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Length of text: ", len(text))

Length of text:  230515


In [27]:
chars = sorted(list(set(text)))
print("".join(chars))
vocab_size = len(chars)



 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]_abcdefghijklmnopqrstuvwxyz{} ·’


In [28]:
stoi = {char:i for i,char in enumerate(chars)}
itos = {i:char for i,char in enumerate(chars)}
encode = lambda sen : [stoi[char] for char in sen]
decode = lambda nums : ''.join([itos[num] for num in nums])

print(encode("hello world"))
print(decode(encode("hello world")))

[71, 68, 75, 75, 78, 1, 86, 78, 81, 75, 67]
hello world


In [29]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

n = int(len(data) * .9)
train_data = data[:n]
val_data = data[n:]

torch.Size([230515]) torch.int64


In [30]:
torch.manual_seed(69420)
batch_size = 4
context_window = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_window, (batch_size,))
    x = torch.stack([data[i:i+context_window] for i in ix])
    y = torch.stack([data[i+1:i+context_window+1] for i in ix])
    return x, y


xb, yb = get_batch("train")
print("inputs")
print(xb.shape)
print(xb)
print("outputs")
print(yb.shape)
print(yb)
print("---")

for b in range(batch_size):
    print(f"For batch {b+1}")
    for c in range(context_window):
        print(f"For input {xb[b, :c+1].tolist()} we have output {yb[b, c]}")

inputs
torch.Size([4, 8])
tensor([[64, 77, 83, 72, 64, 75, 72, 64],
        [83, 28, 13,  7, 80, 84, 78, 83],
        [ 1, 29, 75, 72, 77, 74,  1, 81],
        [ 3,  1, 79, 78, 79, 78, 85, 68]])
outputs
torch.Size([4, 8])
tensor([[77, 83, 72, 64, 75, 72, 64, 82],
        [28, 13,  7, 80, 84, 78, 83, 28],
        [29, 75, 72, 77, 74,  1, 81, 68],
        [ 1, 79, 78, 79, 78, 85, 68, 81]])
---
For batch 1
For input [64] we have output 77
For input [64, 77] we have output 83
For input [64, 77, 83] we have output 72
For input [64, 77, 83, 72] we have output 64
For input [64, 77, 83, 72, 64] we have output 75
For input [64, 77, 83, 72, 64, 75] we have output 72
For input [64, 77, 83, 72, 64, 75, 72] we have output 64
For input [64, 77, 83, 72, 64, 75, 72, 64] we have output 82
For batch 2
For input [83] we have output 28
For input [83, 28] we have output 13
For input [83, 28, 13] we have output 7
For input [83, 28, 13, 7] we have output 80
For input [83, 28, 13, 7, 80] we have output 84
For